##  /data/.home


In [ ]:
%%bash
make_links() {
    local src="${1:-/data/.home}"
    local tgt="${2:-$HOME}"
    
    while IFS= read -r i; do
        rm -fr "$tgt/$(basename "$i")"
        ln -sf "$i" "$tgt/$(basename "$i")"
    done < <(find "$src" -mindepth 1 -maxdepth 1 -type d)
}

# >>>>>>>>>>>>>>>>>>>>使用示例>>>>>>>>>>>>>>>
make_links "/data/.home" "$HOME"


## python & ipykernel

In [ ]:
%%bash
sudo pacman -S --needed --noconfirm python-pip python-ipykernel


## [vscode](../s/vscode.ipynb)


In [ ]:
%%bash
# from https://code.visualstudio.com/Download download tar.gz
# >>>>>>>>>>>>>>>>>>>>下载 VSCode>>>>>>>>>>>>>>>
cd ~/Downloads
curl -L "https://code.visualstudio.com/sha/download?build=stable&os=linux-x64" -o vscode.tar.gz
tar xf vscode.tar.gz
rm vscode.tar.gz

# microsoft 3124568493@qq.com


### gpg

In [ ]:
%%bash
gen_gpg_key() {
    local name="${1:-kefu}"
    local email="${2:-kefu1820@gmail.com}"
    local gd="${GPG_DIR:-$HOME/.gnupg}"
    
    if gpg --list-keys "$email" &>/dev/null; then
        echo "✓ GPG key already exists for $email"
        return
    fi
    
    read -sp "Enter GPG passphrase: " pswd
    echo
    
    chmod 700 "$gd"
    
    local tmp=$(mktemp)
    cat >"$tmp" <<EOF
%echo Generating GPG key
Key-Type: RSA
Key-Length: 4096
Subkey-Type: RSA
Subkey-Length: 4096
Name-Real: $name
Name-Email: $email
Expire-Date: 3y
Passphrase: $pswd
%commit
%echo done
EOF
    
    gpg --batch --generate-key "$tmp"
    rm -f "$tmp"
    find "$gd" -type f -exec chmod 600 {} \;
    
    echo "✓ GPG key generated for $email"
    gpg --list-keys "$email"
}

# >>>>>>>>>>>>>>>>>>>>使用示例>>>>>>>>>>>>>>>
gen_gpg_key "kefu" "kefu1820@gmail.com"


### firefox 

In [ ]:
# kefu51252@gmail.com

## chinese mirrors


In [ ]:
%%bash
set_mirror() {
    local url="${1:-https://mirrors.tuna.tsinghua.edu.cn/manjaro/stable/\$repo/\$arch}"
    sudo tee /etc/pacman.d/mirrorlist <<EOF
# China mirrors tsinghua
Server = $url
EOF
}

# >>>>>>>>>>>>>>>>>>>>使用示例>>>>>>>>>>>>>>>
set_mirror "https://mirrors.tuna.tsinghua.edu.cn/manjaro/stable/\$repo/\$arch"


## yay


In [ ]:
%%bash
sudo pacman -Sy --needed --noconfirm base-devel yay 


In [ ]:
%%bash
cfg_yay() {
    local dir="$HOME/.config/yay"
    mkdir -p "$dir"
    tee "$dir/config.json" > /dev/null <<'EOF'
{
    "editor": "nano",
    "pacmanbin": "pacman",
    "pacmanconf": "/etc/pacman.conf",
    "answerclean": "All",
    "removemake": "ask",
    "maxconcurrentdownloads": 5,
    "cleanAfter": false,
    "batchinstall": true,
    "DevelCheckUpdate": false
}
EOF
    echo "✓ yay config created"
}

# >>>>>>>>>>>>>>>>>>>>使用示例>>>>>>>>>>>>>>>
cfg_yay


## pacman


In [ ]:
%%bash
cfg_pacman() {
    local file="/etc/pacman.conf"
    local -a options=("$@")
    [[ ${#options[@]} -eq 0 ]] && options=("Color" "ILoveCandy" "ParallelDownloads = 5")
    
    for opt in "${options[@]}"; do
        local key=${opt%% *}
        sudo sed -i "/^#\?$key/d; /^\[options\]/a $opt" "$file"
    done
    
    echo "✓ pacman options configured"
}

# >>>>>>>>>>>>>>>>>>>>使用示例>>>>>>>>>>>>>>>
cfg_pacman "Color" "ILoveCandy" "ParallelDownloads = 5"


###  archlinuxcn


In [ ]:
%%bash
add_archlinuxcn() {
    local file="/etc/pacman.conf"
    local server="${1:-https://mirrors.tuna.tsinghua.edu.cn/archlinuxcn/\$arch}"
    
    if ! grep -q "^\[archlinuxcn\]" "$file"; then
        echo -e "\n[archlinuxcn]\nServer = $server" | sudo tee -a "$file"
    else
        sudo sed -i "/^\[archlinuxcn\]/,/^\[/{/^Server/d}" "$file"
        sudo sed -i "/^\[archlinuxcn\]/a Server = $server" "$file"
    fi
    
    echo "✓ archlinuxcn configured: $server"
}

# >>>>>>>>>>>>>>>>>>>>使用示例>>>>>>>>>>>>>>>
add_archlinuxcn "https://mirrors.tuna.tsinghua.edu.cn/archlinuxcn/\$arch"


##  pacman-key


In [ ]:
%%bash
echo "installing pacman-key..."
sudo pacman -S --needed --noconfirm manjaro-keyring archlinux-keyring archlinuxcn-keyring 
sudo pacman-key --init 
sudo pacman-key --populate archlinux manjaro archlinuxcn 
sudo pacman -Syy --noconfirm 


## update system


In [ ]:
%%bash
sudo pacman -Syyu --noconfirm 


In [15]:
%%bash
yay -Syyu --noconfirm 


:: Synchronizing package databases...
 core downloading...
 extra downloading...
 multilib downloading...
 archlinuxcn downloading...
:: Searching AUR for updates...
:: Searching databases for updates...
 there is nothing to do


In [16]:
%%bash
yay -S --needed --noconfirm visual-studio-code-bin 


 -> visual-studio-code-bin-1.108.1-1 is up to date -- skipping
 there is nothing to do


## reboot


## backup

In [17]:
%%bash
create_snapshot() {
    local mnt="${1:-/}"
    local snapshot_name="${2:-update_system}"
    
    # >>>>>>>>>>>>>>>>>>>>处理路径拼接>>>>>>>>>>>>>>>
    [[ "$mnt" == "/" ]] && mnt=""
    local snapshot_dir="${mnt}/.snapshots"
    sudo mkdir -p "$snapshot_dir"
    
    # >>>>>>>>>>>>>>>>>>>>创建快照>>>>>>>>>>>>>>>
    local snapshot_path="$snapshot_dir/$snapshot_name"
    if [[ -d "$snapshot_path" ]]; then
        echo "  ✓ Snapshot already exists: $snapshot_path"
    else
        sudo btrfs subvolume snapshot -r "${mnt:-/}" "$snapshot_path"
        echo "  ✓ Created snapshot: $snapshot_path"
    fi
}

create_snapshot "/" "update_system"

  ✓ Snapshot already exists: /.snapshots/update_system


## google-chrome
## keepassxc
## cryptomator


In [ ]:
%%bash
yay -S --needed --noconfirm google-chrome 
# kefu1820@gmail.com

sudo pacman -S --needed --noconfirm keepassxc 

yay -S --needed --noconfirm cryptomator-bin 


## clash-verge-rev 

In [ ]:
%%bash
sudo pacman -S --needed --noconfirm clash-verge-rev 
# sub link : https://zhuzhuzhu.whtjdasha.com/api/v1/client/subscribe?token=eebe36f8c2eb695b9841a61eb4b03825
# setting : auto start; slient start ; allow lan;

In [ ]:
%%bash
cryptomator &
keepassxc &
clash-verge &
i=$(ip addr show | grep -E 'inet.*global' | awk '{print $2}' | cut -d'/' -f1 | head -n1) && echo "Using IP: $i "
google-chrome-stable --proxy-server="socks5://${i}:7897" &
echo "Done!!!"


## fcitx5


In [ ]:
%%bash
echo "installing fcitx5..."
sudo pacman -S --needed --noconfirm \
	fcitx5 \
	fcitx5-gtk \
	fcitx5-qt \
	fcitx5-configtool \
	fcitx5-chinese-addons \
	fcitx5-pinyin-zhwiki 
kwriteconfig6 --file kwinrc --group Wayland --key 'InputMethod' /usr/share/applications/org.fcitx.Fcitx5.desktop


## git & ssh


###  Git


In [18]:
%%bash
cfg_git() {
    local name="${1:-kefu}"
    local email="${2:-19157521820@163.com}"
    local signingkey=$(gpg --list-secret-keys --keyid-format SHORT "$email" 2>/dev/null | grep sec | awk '{print $2}' | cut -d'/' -f2 | head -1)
    
    git config --global user.name "$name"
    git config --global user.email "$email"
    git config --global init.defaultBranch "main"
    git config --global gpg.program "gpg"
    git config --global user.signingkey "$signingkey"
    git config --global commit.gpgsign "false"
    git config --global credential.helper "store"
    
    echo "✓ Git configured"
}

# >>>>>>>>>>>>>>>>>>>>使用示例>>>>>>>>>>>>>>>
cfg_git "kefu" "19157521820@163.com"


✓ Git configured


###  SSH


In [19]:
%%bash
gen_ssh_key() {
    local email="${1:-19157521820@163.com}"
    local key_type="${2:-ed25519}"
    local ssh_dir="${3:-/data/.home/.ssh}"
    
    mkdir -p "$ssh_dir"
    local ssh_real_path=$(readlink -f "$ssh_dir")
    local ssh_key_path="$ssh_real_path/id_$key_type"
    
    if [[ ! -f "$ssh_key_path" ]]; then
        ssh-keygen -t "$key_type" -C "$email" -f "$ssh_key_path" -N ""
        echo "✓ SSH key generated"
    else
        echo "✓ SSH key already exists"
    fi
    
    chmod 700 "$ssh_real_path"
    chmod 600 "$ssh_real_path"/id_* 2>/dev/null || true
    chmod 644 "$ssh_real_path"/*.pub 2>/dev/null || true
    [[ -f "$ssh_real_path/config" ]] && chmod 600 "$ssh_real_path/config"
    
    rm -rf "$HOME/.ssh"
    ln -sf "$ssh_real_path" "$HOME/.ssh"
}

# >>>>>>>>>>>>>>>>>>>>使用示例>>>>>>>>>>>>>>>
gen_ssh_key "19157521820@163.com" "ed25519" "/data/.home/.ssh"


✓ SSH key already exists


## 配置自动启动


In [ ]:
%%bash
add_autostart() {
    local app="$1"
    local autostart_dir="$HOME/.config/autostart"
    mkdir -p "$autostart_dir"
    
    local exec_path=$(which "$app")
    
    if [[ -n "$exec_path" ]]; then
        cat > "$autostart_dir/$app.desktop" <<EOF
[Desktop Entry]
Type=Application
Name=$app
Exec=$exec_path
Icon=$app
Comment=Auto-start $app
X-GNOME-Autostart-enabled=true
StartupNotify=false
Terminal=false
EOF
        echo "✓ Created autostart for $app"
    else
        echo "✗ Failed to find $app"
    fi
}

# >>>>>>>>>>>>>>>>>>>>使用示例>>>>>>>>>>>>>>>
add_autostart "cryptomator"


## amd gpu

In [ ]:
%%bash
sudo pacman -S --needed --noconfirm \
    mesa \
    lib32-mesa \
    vulkan-radeon \
    lib32-vulkan-radeon \
    libva-mesa-driver \
    lib32-libva-mesa-driver \
    xf86-video-amdgpu \
    rocm-opencl-runtime \
    rocm-hip-runtime


In [ ]:
%%bash
create_snapshot() {
    local mnt="${1:-/}"
    local snapshot_name="${2:-update_system}"
    
    # >>>>>>>>>>>>>>>>>>>>处理路径拼接>>>>>>>>>>>>>>>
    [[ "$mnt" == "/" ]] && mnt=""
    local snapshot_dir="${mnt}/.snapshots"
    sudo mkdir -p "$snapshot_dir"
    
    # >>>>>>>>>>>>>>>>>>>>创建快照>>>>>>>>>>>>>>>
    local snapshot_path="$snapshot_dir/$snapshot_name"
    if [[ -d "$snapshot_path" ]]; then
        echo "  ✓ Snapshot already exists: $snapshot_path"
    else
        sudo btrfs subvolume snapshot -r "${mnt:-/}" "$snapshot_path"
        echo "  ✓ Created snapshot: $snapshot_path"
    fi
}

create_snapshot "/" "amd_gpu"


## ip


In [20]:
%%bash
i=$(ip addr show | grep -E 'inet.*global' | awk '{print $2}' | cut -d'/' -f1 | head -n1) 
echo $i
el="export all_proxy=socks5://${i}:7897"
echo $el

192.168.0.103
export all_proxy=socks5://192.168.0.103:7897
